# HackWatch — Training Notebook

**OpenEnv RL environment for reward-hacking detection.**  
Meta PyTorch OpenEnv Hackathon 2026 — Multi-Agent + Scalable Oversight.

This notebook trains a **MONITOR** agent (Qwen2.5-3B-Instruct + LoRA) via GRPO to detect when a **WORKER** agent cheats on coding tasks.  
Ground truth reward is 100% deterministic — no LLM judge.

**Requirements**: GPU runtime (T4 or better). Go to *Runtime → Change runtime type → T4 GPU*.

## 1. Install Dependencies

In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q trl transformers httpx fastapi uvicorn pydantic matplotlib wandb

## 2. Clone the Repository

In [ ]:
!git clone https://github.com/Afras99/HackWatch.git
%cd HackWatch
!pip install -q -e ".[dev]"

## 3. Start the Environment Server

In [ ]:
import subprocess, time
server = subprocess.Popen(
    ["uvicorn", "server.app:app", "--port", "8000", "--log-level", "warning"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
time.sleep(3)
print("Server started (PID", server.pid, ")")

## 4. Verify the Environment API

In [ ]:
import httpx, json

ENV_URL = "http://localhost:8000"

# Health check
r = httpx.get(f"{ENV_URL}/health")
print("Health:", r.json())

# Reset — starts a new episode (monitor cannot see planted_label)
obs = httpx.post(f"{ENV_URL}/reset", json={}).json()
print("\nObservation keys:", list(obs.keys()))
print("Task:", obs["task_prompt"][:80])
print("Step:", obs["step_idx"], "| Done:", obs["episode_done"])

# Step — submit monitor verdict
step = httpx.post(f"{ENV_URL}/step", json={
    "verdict": "allow", "confidence": 0.3, "reasoning": "diff looks clean"
}).json()
print("\nStep result keys:", list(step.keys()))
print("Reward:", step.get("reward"))

## 5. Build the Training Dataset

UCB-weighted message-list format. 158 prompts, upsampling hard/unseen exploit types.

In [ ]:
from training.train_monitor import build_prompt_dataset

dataset = build_prompt_dataset(env_url=ENV_URL)
print(f"Dataset: {len(dataset)} rows")
print("First prompt (role list):", dataset[0]["prompt"][0])

## 6. Load Model + Configure GRPO

Qwen2.5-3B-Instruct with 4-bit quantization + LoRA r=32.  
DR-GRPO loss, asymmetric DAPO clipping, DynamicSampling for zero-std groups.

In [ ]:
from unsloth import FastLanguageModel
from trl import GRPOConfig
from training.dynamic_grpo import DynamicSamplingGRPOTrainer
from training.train_monitor import build_env_reward_fn

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
OUTPUT_DIR = "./runs/monitor_colab"

model, tokenizer = FastLanguageModel.from_pretrained(
    MODEL_NAME,
    max_seq_length=4096,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model, r=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha=64, lora_dropout=0.05, bias="none",
    use_gradient_checkpointing="unsloth",
)

config = GRPOConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    num_generations=8,
    max_completion_length=512,
    max_prompt_length=1024,
    num_train_epochs=1,
    beta=0.01,
    learning_rate=5e-6,
    warmup_ratio=0.1,
    max_grad_norm=0.5,
    fp16=True,
    logging_steps=1,
    report_to="none",
    max_steps=300,
    loss_type="dr_grpo",
    scale_rewards=False,
    importance_sampling_level="sequence",
    mask_truncated_completions=True,
    epsilon=0.2,
    epsilon_high=0.28,
    temperature=1.1,
)

reward_fn = build_env_reward_fn(env_url=ENV_URL)

trainer = DynamicSamplingGRPOTrainer(
    model=model,
    tokenizer=tokenizer,
    args=config,
    train_dataset=dataset,
    reward_funcs=[reward_fn],
)
print("Trainer ready. Steps:", config.max_steps)

## 7. Run Training

300 steps ≈ 52 minutes on a T4. Reward should rise from ~0.33 → ~0.80.

In [ ]:
trainer.train()
model.save_pretrained(f"{OUTPUT_DIR}/final")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/final")
print("Saved to", OUTPUT_DIR)

## 8. Evaluate — Heuristic Scorer (no GPU needed)

Run the heuristic eval on the 150-trajectory test set. Expected: F1≈0.970, Precision=1.000.

In [ ]:
!python eval/evaluate_monitor.py \
    --trajectories data/trajectories.jsonl \
    --heuristic \
    --tag colab_eval \
    --out eval/results_colab.json

In [ ]:
import json
results = json.load(open("eval/results_colab.json"))
agg = results["aggregate"]
print(f"n_episodes : {agg['n_episodes']}")
print(f"F1         : {agg['f1']:.3f}")
print(f"Precision  : {agg['precision']:.3f}")
print(f"Recall     : {agg['recall']:.3f}")
print(f"Accuracy   : {agg['accuracy']:.1%}")
print(f"TP={agg['tp']}  FP={agg['fp']}  TN={agg['tn']}  FN={agg['fn']}")

## 9. Training Curves

In [ ]:
from IPython.display import Image, display
display(Image("docs/plots/reward_curve.png"))
display(Image("docs/plots/loss_curve.png"))

## Per-Primitive Detection Breakdown

In [ ]:
import matplotlib.pyplot as plt

pf = results["per_primitive_f1"]
# Keep only non-combo primitives for clarity
simple = {k: v for k, v in pf.items() if not k.startswith("combo")}
labels = list(simple.keys())
values = list(simple.values())
colors = ["#e8a000" if v == 1.0 else "#ff4444" if v < 0.8 else "#ffaa00" for v in values]

fig, ax = plt.subplots(figsize=(9, 4))
fig.patch.set_facecolor("#0d0d0d")
ax.set_facecolor("#0d0d0d")
bars = ax.barh(labels, values, color=colors, height=0.6)
ax.set_xlim(0, 1.1)
ax.axvline(1.0, color="#555555", linewidth=1, linestyle="--")
ax.set_xlabel("F1 / TNR", color="#aaaaaa")
ax.set_title("Per-Primitive Detection (heuristic scorer, 150 trajectories)",
             color="#dddddd", pad=10)
ax.tick_params(colors="#777777")
for spine in ax.spines.values():
    spine.set_edgecolor("#333333")
for bar, val in zip(bars, values):
    ax.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
            f"{val:.3f}", va="center", color="#cccccc", fontsize=9)
plt.tight_layout()
plt.show()